
# AI-Powered Video Editing — Motion-Aware Object Replacement (Implementation Notebook)

This notebook implements (and extends) a motion-aware object replacement pipeline for video editing using generative models.

It covers:

- **Object detection + segmentation** (YOLOv8-seg) → per-frame binary masks
- **Trajectory information propagation** (centroid + diameter endpoints) → translation/scale/rotation
- **Optical flow** (RAFT via TorchVision) → motion fields + optional occlusion masks
- **Diffusion inpainting** (Diffusers Stable Diffusion Inpainting) with:
  - **fixed latent noise** (temporal stability)
  - **ControlNet** (depth or canny structural conditioning)
  - **IP-Adapter** (previous frame as visual reference)
- **Frame warping** using optical flow (baseline temporal propagation)
- **Latent warping** (experimental; included for completeness)
- **Proposed extension**: *flow-matching latent search* — search for a latent near a base latent that minimizes optical-flow mismatch between original and inpainted consecutive frames.

> ⚠️ Compute note: Running SD + ControlNet + IP-Adapter on videos is GPU-heavy.
> The notebook is written to be runnable on a single modern GPU (e.g., 12–24GB VRAM) with careful settings.

---

## What you need

- Python 3.10+
- A CUDA GPU (recommended)
- A Hugging Face token if you download models gated by license.




## 0) Install dependencies

Run this once per environment.

If you're in Colab, you may want to restart the runtime after installation.


## Kaggle quick checklist

1) **Turn on GPU**: right sidebar → **Settings** → **Accelerator: GPU**.
2) **Turn on Internet** (needed to download Hugging Face models): right sidebar → **Settings** → toggle **Internet on**.
3) **Add data**: right sidebar → **Add data** → attach a dataset (e.g., DAVIS) or your own MP4.
4) If Stable Diffusion models are gated, add a Hugging Face token via **Add-ons → Secrets** (see the cell below).


In [1]:
# Install deps (Kaggle-friendly)
# Kaggle comes with many preinstalled packages. Sometimes `pip install -U` upgrades shared deps
# (like pillow / pydantic / rich) and pip will warn about conflicts with those preinstalls.
# Those warnings are usually harmless for this notebook, but we pin Pillow to keep common Kaggle
# UI packages (e.g., gradio) happy.
%pip install pillow diffusers transformers accelerate safetensors ultralytics sam2 scikit-image opencv-python imageio imageio-ffmpeg tqdm controlnet-aux

# Optional: if you *also* want to keep gradio/bigframes happy, you can pin these too.
# WARNING: pinning may downgrade versions and could affect other libraries.
# %pip install -q "pydantic<2.12" "rich<14"

# If `pip install sam2` fails in your environment, try installing from source:
# %pip install "git+https://github.com/facebookresearch/sam2.git"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 19.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



## 1) Imports & helpers


In [2]:

import os
import math
import json
import shutil
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image
import cv2
import imageio
from tqdm.auto import tqdm

# YOLOv8 segmentation
from ultralytics import YOLO

# RAFT optical flow (TorchVision)
import torchvision
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights

# Diffusers (Stable Diffusion inpainting + ControlNet)
from diffusers import (
    AutoPipelineForInpainting,
    StableDiffusionInpaintPipeline,
    StableDiffusionControlNetInpaintPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)

# ControlNet auxiliary preprocessors (depth, canny, ...)
from controlnet_aux import MidasDetector, CannyDetector


def torch_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = torch_device()
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE, "dtype:", DTYPE)


def to_pil(img: np.ndarray) -> Image.Image:
    """HWC uint8 RGB -> PIL"""
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    return Image.fromarray(img)


def pil_to_np(pil: Image.Image) -> np.ndarray:
    """PIL -> HWC uint8 RGB"""
    return np.array(pil.convert("RGB"))


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def resize_pad_to_multiple_of_8(pil: Image.Image, target_long_side: int = 768) -> Tuple[Image.Image, Tuple[int,int,int,int]]:
    """Resize keeping aspect ratio so that long side == target_long_side, then pad to multiples of 8.

    Returns: (padded_image, (pad_left, pad_top, pad_right, pad_bottom))
    """
    w, h = pil.size
    scale = target_long_side / max(w, h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    pil_rs = pil.resize((new_w, new_h), Image.BICUBIC)

    # pad to multiple of 8
    pad_w = (8 - (new_w % 8)) % 8
    pad_h = (8 - (new_h % 8)) % 8
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top

    pil_pad = Image.new("RGB", (new_w + pad_w, new_h + pad_h), (0, 0, 0))
    pil_pad.paste(pil_rs, (pad_left, pad_top))

    return pil_pad, (pad_left, pad_top, pad_right, pad_bottom)


def unpad_and_resize(pil: Image.Image, pad: Tuple[int,int,int,int], orig_size: Tuple[int,int]) -> Image.Image:
    pad_left, pad_top, pad_right, pad_bottom = pad
    w, h = pil.size
    cropped = pil.crop((pad_left, pad_top, w - pad_right, h - pad_bottom))
    return cropped.resize(orig_size, Image.BICUBIC)


def mask_from_polygon_list(polys: List[np.ndarray], shape_hw: Tuple[int,int]) -> np.ndarray:
    """polys: list of (N,2) float/ints in image coordinates. Returns binary mask HxW uint8 {0,255}."""
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)
    for poly in polys:
        if poly is None or len(poly) < 3:
            continue
        pts = np.round(poly).astype(np.int32)
        cv2.fillPoly(mask, [pts], 255)
    return mask


def dilate_mask(mask: np.ndarray, k: int = 15, iters: int = 1) -> np.ndarray:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return cv2.dilate(mask, kernel, iterations=iters)


def blur_mask(mask: np.ndarray, k: int = 21) -> np.ndarray:
    k = k if k % 2 == 1 else k + 1
    return cv2.GaussianBlur(mask, (k, k), 0)


def show_grid(frames: List[Image.Image], cols: int = 4, max_items: int = 12, label: Optional[str]=None) -> Image.Image:
    """Make a simple grid preview (PIL)."""
    items = frames[:max_items]
    if not items:
        raise ValueError("No frames")
    w, h = items[0].size
    rows = int(math.ceil(len(items) / cols))
    grid = Image.new("RGB", (cols * w, rows * h), (0,0,0))
    for i, im in enumerate(items):
        r, c = divmod(i, cols)
        grid.paste(im, (c * w, r * h))
    if label:
        # add a small label band on top
        band = Image.new("RGB", (grid.size[0], 40), (20,20,20))
        grid2 = Image.new("RGB", (grid.size[0], grid.size[1] + 40), (0,0,0))
        grid2.paste(band, (0,0))
        grid2.paste(grid, (0,40))
        return grid2
    return grid


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


E0000 00:00:1769080447.743982      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769080447.815842      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769080448.361452      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769080448.361504      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769080448.361507      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769080448.361510      55 computation_placer.cc:177] computation placer already registered. Please check linka

Device: cuda dtype: torch.float16


/usr/local/lib/python3.12/dist-packages/controlnet_aux/segment_anything/modeling/tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_5m_224 in registry with controlnet_aux.segment_anything.modeling.tiny_vit_sam.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/usr/local/lib/python3.12/dist-packages/controlnet_aux/segment_anything/modeling/tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_11m_224 in registry with controlnet_aux.segment_anything.modeling.tiny_vit_sam.tiny_vit_11m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/usr/local/lib/python3.12/dist-packages/controlnet_aux/segment_anything/modeling/tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_21m_224 in registry with controlnet_aux.segment_anything.modeling.tiny_vit_sam.tiny_vit_21m_224. This 


## 2) Datasets (what to evaluate on)

This notebook supports **two evaluation modes**:

1. **Datasets with per-frame masks** (recommended): DAVIS, YouTube-VOS, SegTrack, KITTI-MOTS, VIPSeg, YouTube-VIS.
   - You can use *ground-truth masks* instead of YOLO.

2. **Any custom video** (fallback): we derive masks with YOLOv8-seg.

### Recommended folder convention

- Put datasets under `DATA_ROOT/`.
- Put your experiment outputs under `RUNS_ROOT/`.




### Optical-flow datasets (to validate / stress-test motion components)

If you want to evaluate **the optical-flow piece** (RAFT accuracy, flow-based warping artifacts, and the proposed *flow-matching latent search*), these are common benchmarks:

- **FlyingChairs** (synthetic; classic pretraining dataset)
- **FlyingThings3D** (synthetic; harder motion/occlusion)
- **MPI Sintel** (rendered but closer to real; includes complex motion and occlusions)
- **KITTI Optical Flow 2012/2015** (real driving scenes; sparse-ish ground truth)

> Tip: You can use these datasets purely to test your flow + warping utilities even before you run diffusion inpainting.


In [3]:
# Local paths (works for local Jupyter / Colab)
import zipfile

DATA_ROOT = "."  # folder where frames.zip lives (adjust if needed)

# Our collected dataset (no masks)
FRAMES_ZIP = os.path.join(DATA_ROOT, "frames.zip")
FRAMES_ROOT = os.path.join(DATA_ROOT, "frames")  # after unzip: frames/<category>/<sequence>/<00001.jpg...>

RUNS_ROOT = os.path.join(DATA_ROOT, "runs", "motion_object_replace_frames")
ensure_dir(RUNS_ROOT)

print("DATA_ROOT:", os.path.abspath(DATA_ROOT))
print("FRAMES_ZIP:", os.path.abspath(FRAMES_ZIP))
print("FRAMES_ROOT:", os.path.abspath(FRAMES_ROOT))
print("RUNS_ROOT:", os.path.abspath(RUNS_ROOT))

# Unzip frames.zip once (skip if already extracted)
if os.path.isfile(FRAMES_ZIP) and (not os.path.isdir(FRAMES_ROOT) or len(os.listdir(FRAMES_ROOT)) == 0):
    print("Extracting frames.zip ...")
    with zipfile.ZipFile(FRAMES_ZIP, "r") as zf:
        zf.extractall(DATA_ROOT)
    print("Done. Extracted to:", FRAMES_ROOT)
else:
    print("Found extracted frames folder (or frames.zip missing).")


DATA_ROOT: /kaggle/input/davisthesis
RUNS_ROOT: /kaggle/working/runs/motion_object_replace
Attached inputs:
['davisthesis']



### DAVIS loader (2016/2017)

Expected structure:

```
DAVIS/
  JPEGImages/480p/<seq_name>/<00000.jpg ...>
  Annotations/480p/<seq_name>/<00000.png ...>   # masks (palette PNG)
```

Download instructions are on the official DAVIS site (manual registration may be required).


In [4]:

@dataclass
class VideoSequence:
    name: str
    frames: List[Image.Image]           # RGB
    masks: Optional[List[Image.Image]]  # grayscale or palette; may be None


def load_davis_sequence(davis_root: str, seq_name: str, resolution_dir: str = "Full-Resolution") -> VideoSequence:
    img_dir = os.path.join(davis_root, "JPEGImages", resolution_dir, seq_name)
    ann_dir = os.path.join(davis_root, "Annotations", resolution_dir, seq_name)

    if not os.path.isdir(img_dir):
        raise FileNotFoundError(f"Missing DAVIS frames dir: {img_dir}")

    frame_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png"))])
    frames = [Image.open(os.path.join(img_dir, f)).convert("RGB") for f in frame_files]

    masks = None
    if os.path.isdir(ann_dir):
        mask_files = sorted([f for f in os.listdir(ann_dir) if f.lower().endswith((".png",))])
        if len(mask_files) == len(frame_files):
            masks = [Image.open(os.path.join(ann_dir, f)) for f in mask_files]

    return VideoSequence(name=seq_name, frames=frames, masks=masks)



### Custom video loader

If you have a local video (mp4/mov), set `VIDEO_PATH` and we will extract frames.


In [5]:

VIDEO_PATH = ""  # e.g. "/path/to/video.mp4" (leave empty if using a dataset)


def extract_frames_from_video(video_path: str, max_frames: Optional[int] = None, stride: int = 1) -> List[Image.Image]:
    if not os.path.isfile(video_path):
        raise FileNotFoundError(video_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    frames = []
    idx = 0
    kept = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        if idx % stride == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
            kept += 1
            if max_frames is not None and kept >= max_frames:
                break
        idx += 1
    cap.release()
    return frames

def load_masks_from_dir(masks_dir: str, max_frames: Optional[int] = None, stride: int = 1) -> List[Image.Image]:
    """Load per-frame masks from a directory.

    Expected: files sorted lexicographically correspond to video frames.
    Supports common image formats (png/jpg). Masks will be converted to single-channel 'L'.
    """
    if not os.path.isdir(masks_dir):
        raise FileNotFoundError(masks_dir)

    exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
    files = [f for f in os.listdir(masks_dir) if f.lower().endswith(exts)]
    files.sort()
    if not files:
        raise ValueError(f'No mask images found in {masks_dir}')

    masks = []
    kept = 0
    for idx, fn in enumerate(files):
        if idx % stride != 0:
            continue
        p = os.path.join(masks_dir, fn)
        m = Image.open(p).convert('L')
        masks.append(m)
        kept += 1
        if max_frames is not None and kept >= max_frames:
            break
    return masks


### Frames dataset loader (our collected dataset)

Your dataset is a directory of per-frame JPEGs (no masks), structured like:

```
frames/
  cats/<seq_name>/<00001.jpg ...>
  cars/<seq_name>/<00001.jpg ...>
  humans/<seq_name>/<00001.jpg ...>
```

We'll load a sequence as a list of PIL frames. (Masks will be generated later using **YOLOv8-seg** and/or **SAM2**.)


In [ ]:
from pathlib import Path

def list_frame_sequences(frames_root: str) -> List[str]:
    """Return a list of available sequence directories: frames/<category>/<seq_name>."""
    root = Path(frames_root)
    if not root.exists():
        raise FileNotFoundError(f"frames_root not found: {frames_root}")
    seq_dirs = []
    for cat in sorted([p for p in root.iterdir() if p.is_dir()]):
        for seq in sorted([p for p in cat.iterdir() if p.is_dir()]):
            seq_dirs.append(str(seq))
    return seq_dirs

def load_frames_from_dir(seq_dir: str, max_frames: Optional[int] = None, stride: int = 1) -> List[Image.Image]:
    """Load frames from a directory (sorted numerically by filename stem)."""
    seq_path = Path(seq_dir)
    if not seq_path.is_dir():
        raise FileNotFoundError(seq_dir)

    frame_files = [p for p in seq_path.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    # Sort by numeric stem if possible (e.g. 00001.jpg); else lexicographic
    def _key(p: Path):
        try:
            return int(p.stem)
        except Exception:
            return p.stem
    frame_files = sorted(frame_files, key=_key)

    frames = []
    kept = 0
    for i, p in enumerate(frame_files):
        if i % stride != 0:
            continue
        frames.append(Image.open(p).convert("RGB"))
        kept += 1
        if max_frames is not None and kept >= max_frames:
            break
    return frames

# Quick peek at sequences
if os.path.isdir(FRAMES_ROOT):
    seq_dirs = list_frame_sequences(FRAMES_ROOT)
    print(f"Found {len(seq_dirs)} sequences. Example:\n", "\n ".join(seq_dirs[:5]))
else:
    print("FRAMES_ROOT not found yet. Make sure frames.zip is extracted.")


## 3) Object segmentation / mask generation (YOLOv8-seg and/or SAM2)

- If your dataset provides masks, you can use them directly.
- Otherwise, we infer masks with:
  - **YOLOv8-seg** (per-frame segmentation), and/or
  - **SAM2 video predictor** (prompted once, then tracked through the video).

We also include a **SAM2 vs YOLO** comparison section (SSIM/IoU + a side-by-side overlay video).

**Tip:** YOLO class names follow COCO for standard YOLOv8 models.


In [6]:
# Choose your segmentation / mask generation strategy
# Our `frames` dataset has NO masks, so by default we generate masks with YOLOv8 and/or SAM2.

USE_GT_MASKS = False   # set True only if you provide ground-truth masks
RUN_YOLO_MASKS = True
RUN_SAM2_MASKS = True

# Which masks do we feed into the downstream editing pipeline?
# Options: "yolo" or "sam2" (if both are computed)
MASK_SOURCE_FOR_PIPELINE = "sam2"

# -----------------------
# YOLOv8-seg config
# -----------------------
YOLO_MODEL = "yolov8x-seg.pt"  # smaller/faster: yolov8n-seg.pt
YOLO_CONF = 0.25

# COCO class name, e.g. "person", "car", "dog", "cat"
TARGET_CLASS_NAME = "person"

# Optional auto-pick for the frames dataset categories (if FRAMES_CATEGORY is defined later)
CATEGORY_TO_COCO_CLASS = {"humans": "person", "cars": "car", "cats": "cat"}
if "FRAMES_CATEGORY" in globals():
    TARGET_CLASS_NAME = CATEGORY_TO_COCO_CLASS.get(FRAMES_CATEGORY, TARGET_CLASS_NAME)

print("TARGET_CLASS_NAME:", TARGET_CLASS_NAME)

# -----------------------
# SAM2 config
# -----------------------
# We use SAM2's video predictor, prompted by bounding boxes from YOLO on the prompt frame.
# This makes the pipeline fully automatic (no manual clicks required).

SAM2_LOAD_METHOD = "hf"  # "hf" (recommended) or "local"

# HuggingFace model IDs (pick one):
SAM2_HF_MODEL_ID = "facebook/sam2-hiera-base-plus"  # good default
# SAM2_HF_MODEL_ID = "facebook/sam2-hiera-large"    # higher quality, heavier

# If using SAM2_LOAD_METHOD="local", set these (see facebookresearch/sam2 README):
SAM2_CHECKPOINT_PATH = "./checkpoints/sam2.1_hiera_large.pt"
SAM2_MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"

# Prompting strategy
SAM2_PROMPT_FRAME_IDX = 0
SAM2_MAX_OBJECTS = 5  # track up to N objects (from YOLO detections on the prompt frame)

# -----------------------
# mask postprocess (used for soft masks in inpainting)
# -----------------------
DILATE_K = 21
DILATE_ITERS = 1
BLUR_K = 21


In [7]:
# NOTE:
# YOLO initialization (yolo = YOLO(...)) and TARGET_CLASS_ID are performed later in the
# "Video run config" section *after* you choose FRAMES_CATEGORY / TARGET_CLASS_NAME.
# This makes it easy to switch experiments (e.g., person->drone, cat->tiger) without
# having to restart the kernel.


Target class: person id: 0


In [8]:

@torch.inference_mode()
def yolo_segment_frames(frames: List[Image.Image], target_class_id: int, conf: float = 0.25) -> List[Image.Image]:
    """Return a list of PIL (L) masks matching the target class.

    Strategy: For each frame, keep polygons for detections of the target class and OR them together.
    """
    masks_pil = []

    for pil in tqdm(frames, desc="YOLOv8-seg"):
        img = pil_to_np(pil)  # RGB
        # Ultralytics accepts numpy RGB
        res = yolo.predict(img, conf=conf, verbose=False)[0]

        polys = []
        if res.masks is not None and res.boxes is not None:
            cls = res.boxes.cls.detach().cpu().numpy().astype(int)
            # res.masks.xy is a list of polygons in xy
            for i, c in enumerate(cls):
                if int(c) != int(target_class_id):
                    continue
                poly = res.masks.xy[i]
                polys.append(np.array(poly, dtype=np.float32))

        mask = mask_from_polygon_list(polys, shape_hw=img.shape[:2])
        mask = dilate_mask(mask, k=DILATE_K, iters=DILATE_ITERS)
        mask = blur_mask(mask, k=BLUR_K)
        masks_pil.append(Image.fromarray(mask).convert("L"))

    return masks_pil


### SAM2 video masks (prompted by YOLO boxes)

SAM2 needs prompts (points/boxes/masks). Since our dataset has no annotations, we use **YOLOv8 detections on the prompt frame** to create bounding-box prompts, then let **SAM2 track/propagate** masks through the whole video.

This yields a per-frame mask sequence comparable to YOLOv8-seg masks.


In [ ]:
from contextlib import nullcontext

def yolo_boxes_from_frame(
    pil: Image.Image,
    target_class_id: int,
    conf: float = 0.25,
    max_boxes: int = 5,
) -> List[np.ndarray]:
    """Run YOLO on a single frame and return up to N xyxy boxes for the target class (sorted by confidence)."""
    img = pil_to_np(pil)  # RGB
    res = yolo.predict(img, conf=conf, verbose=False)[0]

    boxes = []
    if res.boxes is not None and len(res.boxes) > 0:
        cls = res.boxes.cls.detach().cpu().numpy().astype(int)
        confs = res.boxes.conf.detach().cpu().numpy()
        xyxy = res.boxes.xyxy.detach().cpu().numpy()
        for c, cf, bb in zip(cls, confs, xyxy):
            if int(c) != int(target_class_id):
                continue
            boxes.append((float(cf), bb.astype(np.float32)))

    boxes = sorted(boxes, key=lambda x: -x[0])[:max_boxes]
    return [b for _, b in boxes]


def write_frames_to_dir(frames: List[Image.Image], out_dir: str) -> List[str]:
    """Save PIL frames to a directory as numbered JPEGs and return the filenames."""
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    ensure_dir(out_dir)

    frame_names = []
    for i, pil in enumerate(frames):
        name = f"{i:05d}.jpg"
        path = os.path.join(out_dir, name)
        pil.save(path, quality=95)
        frame_names.append(name)
    return frame_names


def load_sam2_video_predictor():
    """Load SAM2 video predictor (either from HF or from local cfg+checkpoint)."""
    if SAM2_LOAD_METHOD == "hf":
        from sam2.sam2_video_predictor import SAM2VideoPredictor
        predictor = SAM2VideoPredictor.from_pretrained(SAM2_HF_MODEL_ID)
    else:
        from sam2.build_sam import build_sam2_video_predictor
        predictor = build_sam2_video_predictor(SAM2_MODEL_CFG, SAM2_CHECKPOINT_PATH)

    # Move to device if possible
    try:
        predictor.to(DEVICE)
    except Exception:
        pass

    return predictor


@torch.inference_mode()
def sam2_segment_frames_with_yolo_prompts(
    frames: List[Image.Image],
    prompt_frame_idx: int,
    target_class_id: int,
    yolo_conf: float,
    max_objects: int = 5,
) -> Tuple[List[Image.Image], List[np.ndarray]]:
    """Generate a per-frame mask sequence using SAM2, prompted by YOLO boxes on one frame.

    Returns:
      masks_sam2: list of PIL 'L' masks (soft, after dilation+blur, aligned to frames)
      prompt_boxes: list of xyxy boxes used as SAM2 prompts
    """
    # 1) get YOLO boxes on prompt frame
    prompt_boxes = yolo_boxes_from_frame(
        frames[prompt_frame_idx], target_class_id, conf=yolo_conf, max_boxes=max_objects
    )
    if len(prompt_boxes) == 0:
        raise RuntimeError(
            "YOLO found 0 boxes for the target class on the prompt frame. "

            "Try lowering YOLO_CONF or changing TARGET_CLASS_NAME."
        )

    # 2) write frames to a temp dir (SAM2 video predictor expects a video_path directory)
    tmp_dir = os.path.join(RUNS_ROOT, "_tmp_sam2_video_frames")
    write_frames_to_dir(frames, tmp_dir)

    # 3) load predictor + init state
    predictor = load_sam2_video_predictor()

    amp_ctx = (
        torch.autocast("cuda", dtype=torch.bfloat16)
        if DEVICE == "cuda"
        else nullcontext()
    )

    n = len(frames)
    masks = [None] * n

    with torch.inference_mode(), amp_ctx:
        inference_state = predictor.init_state(video_path=tmp_dir)

        # Add each prompt box as a separate object to track
        for j, box in enumerate(prompt_boxes):
            obj_id = int(j + 1)  # unique object id
            predictor.add_new_points_or_box(
                inference_state=inference_state,
                frame_idx=prompt_frame_idx,
                obj_id=obj_id,
                box=box,
            )

        # Propagate through the video
        for frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
            # Union all tracked objects into a single mask for this frame
            union = None
            for k, oid in enumerate(out_obj_ids):
                logits = out_mask_logits[k]
                # logits can be [1,H,W] or [H,W]
                if logits.ndim == 3:
                    logits2 = logits[0]
                else:
                    logits2 = logits
                m = (logits2 > 0.0).detach().cpu().numpy().astype(np.uint8) * 255
                union = m if union is None else np.maximum(union, m)

            if union is None:
                # no objects -> empty mask
                w, h = frames[0].size
                union = np.zeros((h, w), dtype=np.uint8)

            # same postprocess used by YOLO masks (soft edges help inpainting)
            union = dilate_mask(union, k=DILATE_K, iters=DILATE_ITERS)
            union = blur_mask(union, k=BLUR_K)

            masks[frame_idx] = Image.fromarray(union).convert("L")

    # Fill any missing frames with empty masks
    for i in range(n):
        if masks[i] is None:
            w, h = frames[i].size
            masks[i] = Image.fromarray(np.zeros((h, w), dtype=np.uint8)).convert("L")

    return masks, prompt_boxes



## 4) Trajectory information propagation (centroid + diameter endpoints)

This is the thesis' basic geometric motion transfer:

- Find two farthest boundary points on a mask (approx. diameter endpoints)
- Midpoint is centroid
- Between frames: estimate translation, scale ratio, and rotation

We implement it for analysis and as a baseline compositor.


In [9]:

from itertools import combinations


def mask_to_contour(mask_01: np.ndarray) -> Optional[np.ndarray]:
    """mask_01: HxW bool or {0,1}.
    Returns contour points (N,2) in xy or None.
    """
    mask_u8 = (mask_01.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    # choose largest contour
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt.reshape(-1, 2)  # xy
    return pts


def farthest_points(pts_xy: np.ndarray, max_samples: int = 2000) -> Tuple[np.ndarray, np.ndarray]:
    """Brute force farthest pair (with optional random subsampling for speed)."""
    n = pts_xy.shape[0]
    if n > max_samples:
        idx = np.random.choice(n, max_samples, replace=False)
        pts = pts_xy[idx]
    else:
        pts = pts_xy

    # compute farthest pair
    max_d = -1.0
    p1 = pts[0]
    p2 = pts[-1]
    for a, b in combinations(range(len(pts)), 2):
        d = np.sum((pts[a] - pts[b]) ** 2)
        if d > max_d:
            max_d = d
            p1, p2 = pts[a], pts[b]
    return p1.astype(np.float32), p2.astype(np.float32)


def trajectory_features(mask_pil: Image.Image) -> Optional[Dict[str, Any]]:
    """Extract centroid, diameter endpoints, diameter length, angle."""
    m = np.array(mask_pil.convert("L"))
    # binarize
    m01 = (m > 128).astype(np.uint8)
    pts = mask_to_contour(m01)
    if pts is None or len(pts) < 10:
        return None

    p1, p2 = farthest_points(pts)
    centroid = (p1 + p2) / 2.0
    diam = float(np.linalg.norm(p2 - p1))
    angle = float(math.atan2(p2[1] - p1[1], p2[0] - p1[0]))  # radians

    return {"p1": p1, "p2": p2, "centroid": centroid, "diameter": diam, "angle": angle}


def relative_transform(f1: Dict[str, Any], f2: Dict[str, Any]) -> Dict[str, float]:
    """Compute translation, scale ratio, rotation (radians) from frame1 -> frame2."""
    dx, dy = (f2["centroid"] - f1["centroid"]).tolist()
    # size ratio based on squared diameter (as in thesis)
    r = (f2["diameter"] ** 2) / (f1["diameter"] ** 2 + 1e-8)
    dtheta = float(f2["angle"] - f1["angle"])
    # normalize angle to [-pi, pi]
    dtheta = (dtheta + math.pi) % (2 * math.pi) - math.pi
    return {"dx": float(dx), "dy": float(dy), "scale": float(math.sqrt(max(r, 1e-8))), "dtheta": dtheta}



## 5) Optical flow with RAFT (TorchVision)

We use `torchvision.models.optical_flow.raft_large` with pretrained weights.

Outputs:
- dense flow field **(H,W,2)** in pixel units
- optional occlusion mask (forward-backward consistency)


In [10]:

@torch.inference_mode()
def load_raft() -> Tuple[torch.nn.Module, Any]:
    weights = Raft_Large_Weights.DEFAULT
    model = raft_large(weights=weights).to(DEVICE)
    model.eval()
    transforms = weights.transforms()
    return model, transforms


def pil_to_raft_tensor(pil: Image.Image) -> torch.Tensor:
    # raft weights transforms expect list of tensors? We'll use the built-in transforms later.
    arr = np.array(pil.convert("RGB"))
    ten = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
    return ten


def flow_to_numpy(flow: torch.Tensor) -> np.ndarray:
    """flow: (1,2,H,W) -> (H,W,2) float32"""
    flow = flow[0].permute(1, 2, 0).detach().cpu().numpy().astype(np.float32)
    return flow


def compute_occlusion_mask(flow_fwd: np.ndarray, flow_bwd: np.ndarray, thresh: float = 1.0) -> np.ndarray:
    """Forward-backward consistency check.

    occluded=True where ||fwd + warp(bwd, fwd)|| > thresh

    Returns: HxW bool
    """
    h, w, _ = flow_fwd.shape
    # warp backward flow into frame1 coords using forward flow
    bwd_warp = warp_flow(flow_bwd, flow_fwd)
    fb = flow_fwd + bwd_warp
    err = np.linalg.norm(fb, axis=2)
    return err > thresh


def warp_flow(img_or_flow: np.ndarray, flow: np.ndarray) -> np.ndarray:
    """Backward-warp img_or_flow using flow.

    If flow is from target->source, and img_or_flow is source, then dest(x)=source(x+flow(x)).

    Here we use cv2.remap: dest(x)=src(map_x, map_y)
    """
    h, w = flow.shape[:2]
    grid_x, grid_y = np.meshgrid(np.arange(w), np.arange(h))
    map_x = (grid_x + flow[..., 0]).astype(np.float32)
    map_y = (grid_y + flow[..., 1]).astype(np.float32)

    if img_or_flow.ndim == 2:
        return cv2.remap(img_or_flow, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    elif img_or_flow.ndim == 3 and img_or_flow.shape[2] in (2,3,4):
        return cv2.remap(img_or_flow, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    else:
        raise ValueError(f"Unexpected shape: {img_or_flow.shape}")


@torch.inference_mode()
def raft_flow_between(pil1: Image.Image, pil2: Image.Image, raft_model, raft_transforms) -> np.ndarray:
    # Convert to tensors
    t1 = pil_to_raft_tensor(pil1)
    t2 = pil_to_raft_tensor(pil2)

    # transforms handle resizing / normalization
    t1, t2 = raft_transforms(t1, t2)
    t1 = t1.to(DEVICE)
    t2 = t2.to(DEVICE)

    flow_list = raft_model(t1[None], t2[None])
    flow = flow_list[-1]
    return flow_to_numpy(flow)


raft_model, raft_transforms = load_raft()
print("RAFT loaded")


Downloading: "https://download.pytorch.org/models/raft_large_C_T_SKHT_V2-ff5fadd5.pth" to /root/.cache/torch/hub/checkpoints/raft_large_C_T_SKHT_V2-ff5fadd5.pth


100%|██████████| 20.1M/20.1M [00:00<00:00, 128MB/s] 


RAFT loaded



## 6) Control images (depth + canny)

- Depth maps: MiDaS via `controlnet_aux.MidasDetector`
- Canny edges: `controlnet_aux.CannyDetector` (pure CV)


In [11]:

# Preprocessors
'''midas = MidasDetector.from_pretrained("lllyasviel/Annotators")
canny = CannyDetector()


def make_depth_control(pil: Image.Image, detect_resolution: int = 512, image_resolution: int = 512) -> Image.Image:
    # midas returns a PIL (RGB) depth visualization
    # controlnet_aux typically expects multiples of 64 for best results
    img = pil.resize((image_resolution, image_resolution), Image.BICUBIC)
    depth = midas(img, detect_resolution=detect_resolution, image_resolution=image_resolution)
    return depth
def _ensure_same_size(control: Image.Image, ref: Image.Image, resample=Image.BICUBIC) -> Image.Image:
     """Diffusers ControlNet expects control_image to match the input image size exactly."""
     if control.size != ref.size:
         control = control.resize(ref.size, resample)
     return control


def make_depth_control(
    pil: Image.Image,
    detect_resolution: int = 512,
    image_resolution: Optional[int] = None,
    ) -> Image.Image:
    w, h = pil.size
    pil_rgb = pil.convert("RGB")

     # Let controlnet-aux handle aspect ratio internally.
    if image_resolution is None:
        image_resolution = max(w, h)

    depth = midas(pil_rgb, detect_resolution=detect_resolution, image_resolution=image_resolution)
    if isinstance(depth, np.ndarray):
        depth = Image.fromarray(depth)
    depth = depth.convert("RGB")
    depth = _ensure_same_size(depth, pil_rgb, resample=Image.BICUBIC)
    return depth


def make_canny_control(pil: Image.Image, low: int = 100, high: int = 200) -> Image.Image:
    #img = pil.resize((512, 512), Image.BICUBIC)
    #edges = canny(img, low_threshold=low, high_threshold=high)
    img = pil.convert("RGB")
    edges = canny(img, low_threshold=low, high_threshold=high)
    if isinstance(edges, np.ndarray):
        edges = Image.fromarray(edges)
    edges = edges.convert("RGB")
    edges = _ensure_same_size(edges, img, resample=Image.NEAREST)
    return edges
    #return edges.convert("RGB")'''

# Preprocessors
midas = MidasDetector.from_pretrained("lllyasviel/Annotators")
canny = CannyDetector()

def _ensure_same_size(control: Image.Image, ref: Image.Image, resample=Image.BICUBIC) -> Image.Image:
    """Diffusers ControlNet expects control_image to match the input image size exactly."""
    if control.size != ref.size:
        control = control.resize(ref.size, resample)
    return control

def make_depth_control(
    pil: Image.Image,
    detect_resolution: int = 512,
    image_resolution: Optional[int] = None,
) -> Image.Image:
    """Generate a depth control image that matches `pil.size` exactly.

    We run MiDaS at a possibly smaller internal resolution for speed, then resize back to the
    frame resolution so ControlNet tensors match.
    """
    w, h = pil.size
    pil_rgb = pil.convert("RGB")

    # Let controlnet-aux handle aspect ratio internally.
    if image_resolution is None:
        image_resolution = max(w, h)

    depth = midas(pil_rgb, detect_resolution=detect_resolution, image_resolution=image_resolution)

    if isinstance(depth, np.ndarray):
        depth = Image.fromarray(depth)

    depth = depth.convert("RGB")
    depth = _ensure_same_size(depth, pil_rgb, resample=Image.BICUBIC)
    return depth

def make_canny_control(pil: Image.Image, low: int = 100, high: int = 200) -> Image.Image:
    """Generate a canny control image that matches `pil.size` exactly."""
    img = pil.convert("RGB")
    edges = canny(img, low_threshold=low, high_threshold=high)
    if isinstance(edges, np.ndarray):
        edges = Image.fromarray(edges)

    edges = edges.convert("RGB")
    # Canny is edge-like; avoid blurring edges if a resize is needed
    edges = _ensure_same_size(edges, img, resample=Image.NEAREST)
    return edges

dpt_hybrid-midas-501f0c75.pt:   0%|          | 0.00/493M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(



## 7) Diffusion Inpainting Pipelines

We provide two options:

1. **StableDiffusionInpaintPipeline** (no ControlNet)
2. **StableDiffusionControlNetInpaintPipeline** (ControlNet depth or canny)

And optionally load **IP-Adapter** for visual reference conditioning.

> Notes
> - Diffusers recommends using inpainting-tuned checkpoints.
> - Inpainting pipeline supports `ip_adapter_image` (visual reference) and `latents` (fixed noise).


In [12]:

# ---- Model IDs (change if you prefer SDXL) ----
INPAINT_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"  # mirror of runwayml inpainting

# ControlNet options
CONTROLNET_TYPE = "depth"  # "depth" or "canny" or "inpaint" (see ControlNet v1.1)
CONTROLNET_MODEL_ID = {
    "depth": "lllyasviel/sd-controlnet-depth",
    "canny": "lllyasviel/sd-controlnet-canny",
    "inpaint": "lllyasviel/control_v11p_sd15_inpaint",
}[CONTROLNET_TYPE]

USE_CONTROLNET = True
USE_IP_ADAPTER = True

# IP-Adapter weights
IP_ADAPTER_REPO = "h94/IP-Adapter"
IP_ADAPTER_SUBFOLDER = "models"
IP_ADAPTER_WEIGHT = "ip-adapter_sd15.safetensors"  # general image reference
IP_ADAPTER_SCALE = 0.6

# Inpainting sampling params
NUM_INFERENCE_STEPS = 25
GUIDANCE_SCALE = 7.0
STRENGTH = 0.95  # how much to modify masked region
SEED = 123

# Output
RUN_NAME = "demo_run"
OUT_DIR = os.path.join(RUNS_ROOT, RUN_NAME)
ensure_dir(OUT_DIR)
print("OUT_DIR:", OUT_DIR)


OUT_DIR: /kaggle/working/runs/motion_object_replace/demo_run


In [13]:

import inspect


def load_inpaint_pipeline() -> Any:
    if USE_CONTROLNET:
        controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL_ID, torch_dtype=DTYPE)
        pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
            INPAINT_MODEL_ID,
            controlnet=controlnet,
            torch_dtype=DTYPE,
            safety_checker=None,
        )
    else:
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            INPAINT_MODEL_ID,
            torch_dtype=DTYPE,
            safety_checker=None,
        )

    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(DEVICE)

    # Memory optimizations
    if hasattr(pipe, "enable_xformers_memory_efficient_attention"):
        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception as e:
            print("xFormers not enabled:", e)

    if hasattr(pipe, "enable_model_cpu_offload") and DEVICE == "cuda":
        # Uncomment if you're OOM; it can slow things down.
        # pipe.enable_model_cpu_offload()
        pass

    if USE_IP_ADAPTER:
        # Load image prompt adapter
        pipe.load_ip_adapter(IP_ADAPTER_REPO, subfolder=IP_ADAPTER_SUBFOLDER, weight_name=IP_ADAPTER_WEIGHT)
        pipe.set_ip_adapter_scale(IP_ADAPTER_SCALE)

    return pipe


def pipe_call_inpaint(pipe, prompt: str, image: Image.Image, mask_image: Image.Image,
                      control_image: Optional[Image.Image] = None,
                      ip_adapter_image: Optional[Image.Image] = None,
                      latents: Optional[torch.Tensor] = None,
                      generator: Optional[torch.Generator] = None,
                      **kwargs):
    """Call pipe robustly across diffusers versions."""
    sig = inspect.signature(pipe.__call__)
    kw = dict(
        prompt=prompt,
        image=image,
        mask_image=mask_image,
        strength=kwargs.pop("strength", STRENGTH),
        num_inference_steps=kwargs.pop("num_inference_steps", NUM_INFERENCE_STEPS),
        guidance_scale=kwargs.pop("guidance_scale", GUIDANCE_SCALE),
        generator=generator,
        latents=latents,
    )

    '''if USE_IP_ADAPTER and ip_adapter_image is not None and "ip_adapter_image" in sig.parameters:
        kw["ip_adapter_image"] = ip_adapter_image'''
    if USE_IP_ADAPTER and "ip_adapter_image" in sig.parameters:
        if ip_adapter_image is None:
            raise ValueError(
                "USE_IP_ADAPTER=True but ip_adapter_image=None. "
                "Provide a reference image (e.g., the current frame for t==0) "
                "or set USE_IP_ADAPTER=False."
            )
        kw["ip_adapter_image"] = ip_adapter_image

    if USE_CONTROLNET and control_image is not None:
        # Different versions may call this differently
        if "control_image" in sig.parameters:
            kw["control_image"] = control_image
        elif "controlnet_conditioning_image" in sig.parameters:
            kw["controlnet_conditioning_image"] = control_image
        else:
            # fall back to positional
            return pipe(prompt, image, mask_image, control_image, **kw)

    kw.update(kwargs)
    return pipe(**kw)


pipe = load_inpaint_pipeline()
print("Loaded pipeline:", type(pipe).__name__)


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

...

xFormers not enabled: Refer to https://github.com/facebookresearch/xformers for more information on how to install xformers


models/ip-adapter_sd15.safetensors:   0%|          | 0.00/44.6M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

models/image_encoder/model.safetensors:   0%|          | 0.00/2.53G [00:00<?, ?B/s]

Loaded pipeline: StableDiffusionControlNetInpaintPipeline



## 8) Fixed latent noise (temporal consistency)

We create a **single base latent tensor** and reuse it for each frame.

- This reduces frame-to-frame stochastic variation.
- You can optionally **perturb** it per frame (for the flow-matching search).


In [14]:

# Create a fixed generator
gen = torch.Generator(device=DEVICE)
gen.manual_seed(SEED)


def make_base_latents(pipe, height: int, width: int, generator: torch.Generator) -> torch.Tensor:
    # Diffusers uses VAE latent channels (typically 4)
    latent_channels = getattr(pipe.vae.config, "latent_channels", 4)
    latents_shape = (1, latent_channels, height // 8, width // 8)
    latents = torch.randn(latents_shape, generator=generator, device=DEVICE, dtype=DTYPE)
    return latents



## 9) Video run config

Pick **either** a DAVIS sequence **or** a custom video.

Also set the replacement prompt.


In [15]:
# --- Choose input ---
# We added support for the collected `frames` dataset (no masks).
# You can still use DAVIS or a custom MP4 if you want.

# ---------------------------
# Quick experiment presets
# ---------------------------
# Pick one:
#   - "human_to_drone": replace a person with a drone
#   - "cat_to_tiger"  : replace a cat with a tiger
#   - "custom"        : set everything manually below
EXPERIMENT_PRESET = "custom"

# ---------------------------
# Defaults (manual edit)
# ---------------------------
USE_FRAMES_DATASET = True
FRAMES_CATEGORY = "humans"  # one of: cats, cars, humans
FRAMES_SEQ = "humans__vid3_960x540_30fps_10s"  # folder name inside frames/<category>/

# Prompts (used by the inpainting pipeline later)
PROMPT = "a cute robot"
NEG_PROMPT = "blurry, low quality, worst quality, jpeg artifacts, deformed, ugly"

# ---------------------------
# Apply preset overrides
# ---------------------------
if EXPERIMENT_PRESET == "human_to_drone":
    FRAMES_CATEGORY = "humans"
    FRAMES_SEQ = "humans__vid3_960x540_30fps_10s"
    PROMPT = "a realistic quadcopter drone hovering, matching the scene lighting and perspective, high detail"
    NEG_PROMPT = "person, human, face, hands, arms, legs, body, blurry, low quality, text, watermark"
elif EXPERIMENT_PRESET == "cat_to_tiger":
    FRAMES_CATEGORY = "cats"
    FRAMES_SEQ = "cats__vid1_960x540_30fps"
    PROMPT = "a realistic tiger, full body, same pose and perspective, detailed fur, natural lighting"
    NEG_PROMPT = "domestic cat, kitten, collar, blurry, low quality, text, watermark"

FRAMES_SEQ_DIR = os.path.join(FRAMES_ROOT, FRAMES_CATEGORY, FRAMES_SEQ)

# Fallback options (keep for debugging)
USE_DAVIS = False
DAVIS_ROOT = os.path.join(DATA_ROOT, "DAVIS")
DAVIS_SEQ = "bear"

VIDEO_PATH = ""  # e.g. "/path/to/video.mp4" (leave empty if using frames dataset)

# Optional: if you use a custom VIDEO_PATH and you already have per-frame masks,
# upload them as a folder and set MASKS_DIR to that folder.
MASKS_DIR = ""

MAX_FRAMES = 60   # set to 300 for full 10s clips
FRAME_STRIDE = 1

# Processing resolution (long side), keep aspect, then pad to /8
TARGET_LONG_SIDE = 768
# Fallbacks if you ran this cell before the segmentation config cell
if "TARGET_CLASS_NAME" not in globals():
    TARGET_CLASS_NAME = "person"
if "YOLO_MODEL" not in globals():
    YOLO_MODEL = "yolov8x-seg.pt"
if "CATEGORY_TO_COCO_CLASS" not in globals():
    CATEGORY_TO_COCO_CLASS = {"humans": "person", "cars": "car", "cats": "cat"}


# ---------------------------
# IMPORTANT: choose which ORIGINAL object to segment
# (we segment the source object, not the replacement)
# ---------------------------
if "CATEGORY_TO_COCO_CLASS" in globals():
    TARGET_CLASS_NAME = CATEGORY_TO_COCO_CLASS.get(FRAMES_CATEGORY, TARGET_CLASS_NAME)

print("FRAMES_SEQ_DIR:", FRAMES_SEQ_DIR)
print("TARGET_CLASS_NAME (source object):", TARGET_CLASS_NAME)

# (Re)initialize YOLO and class id based on TARGET_CLASS_NAME
yolo = YOLO(YOLO_MODEL)
name_to_id = {v: k for k, v in yolo.model.names.items()}
if TARGET_CLASS_NAME not in name_to_id:
    print("Available classes:", list(name_to_id.keys())[:25], "...")
    raise ValueError(f"TARGET_CLASS_NAME='{TARGET_CLASS_NAME}' not found in YOLO classes")
TARGET_CLASS_ID = name_to_id[TARGET_CLASS_NAME]
print("Target class id:", TARGET_CLASS_ID)


In [ ]:
# Load sequence
gt_masks = None

if USE_FRAMES_DATASET:
    if not os.path.isdir(FRAMES_SEQ_DIR):
        print("FRAMES_SEQ_DIR not found:", FRAMES_SEQ_DIR)
        print("Available sequences under FRAMES_ROOT:")
        for p in list_frame_sequences(FRAMES_ROOT)[:20]:
            print(" -", p)
        raise FileNotFoundError(FRAMES_SEQ_DIR)

    frames = load_frames_from_dir(FRAMES_SEQ_DIR, max_frames=MAX_FRAMES, stride=FRAME_STRIDE)

elif USE_DAVIS:
    seq = load_davis_sequence(DAVIS_ROOT, DAVIS_SEQ)
    frames = seq.frames[:MAX_FRAMES]
    gt_masks = seq.masks[:MAX_FRAMES] if seq.masks is not None else None

else:
    if not VIDEO_PATH:
        raise ValueError("Set VIDEO_PATH, or enable USE_FRAMES_DATASET / USE_DAVIS")
    frames = extract_frames_from_video(VIDEO_PATH, max_frames=MAX_FRAMES, stride=FRAME_STRIDE)
    if MASKS_DIR:
        gt_masks = load_masks_from_dir(MASKS_DIR, max_frames=MAX_FRAMES, stride=FRAME_STRIDE)
        print("Loaded custom masks from:", MASKS_DIR, "count=", len(gt_masks))

print("Frames:", len(frames), "GT masks:", gt_masks is not None)
print("Original size:", frames[0].size)

# Preview a few
preview = show_grid(frames[:12], cols=4)
preview


Frames: 30 GT masks: True
Original size: (1920, 1080)



## 10) Prepare frames/masks at model resolution

We resize & pad each frame to preserve aspect ratio and align sizes to multiples of 8.
We do the same for masks.


In [ ]:

# Resize/pad frames
frames_proc = []
pads = []
orig_sizes = []

for pil in frames:
    orig_sizes.append(pil.size)
    pil_pad, pad = resize_pad_to_multiple_of_8(pil, target_long_side=TARGET_LONG_SIDE)
    frames_proc.append(pil_pad)
    pads.append(pad)

print("Processed size:", frames_proc[0].size, "pad:", pads[0])
show_grid(frames_proc[:12], cols=4)


In [ ]:
# Prepare masks
# We compute (optionally):
# - masks_yolo: YOLOv8-seg masks (per-frame)
# - masks_sam2: SAM2 video-tracked masks (prompted by YOLO boxes on one frame)
#
# Then we pick a single `masks_proc` to feed into the downstream editing pipeline.

masks_gt_proc = None
masks_yolo = None
masks_sam2 = None
sam2_prompt_boxes = None

# 1) GT masks (if you have them)
if USE_GT_MASKS and gt_masks is not None:
    masks_gt_proc = []
    for m_pil, pad, orig in zip(gt_masks, pads, orig_sizes):
        # Convert GT to binary mask (any non-zero label)
        m_np = np.array(m_pil)
        m_bin = (m_np > 0).astype(np.uint8) * 255
        m_pil_bin = Image.fromarray(m_bin).convert("L")
        m_pad, _ = resize_pad_to_multiple_of_8(m_pil_bin.convert("RGB"), target_long_side=TARGET_LONG_SIDE)
        masks_gt_proc.append(m_pad.convert("L"))

# 2) YOLO masks (no GT available)
if RUN_YOLO_MASKS:
    masks_yolo = yolo_segment_frames(frames_proc, TARGET_CLASS_ID, conf=YOLO_CONF)

# 3) SAM2 masks (prompted by YOLO boxes on the prompt frame)
if RUN_SAM2_MASKS:
    try:
        masks_sam2, sam2_prompt_boxes = sam2_segment_frames_with_yolo_prompts(
            frames=frames_proc,
            prompt_frame_idx=SAM2_PROMPT_FRAME_IDX,
            target_class_id=TARGET_CLASS_ID,
            yolo_conf=YOLO_CONF,
            max_objects=SAM2_MAX_OBJECTS,
        )
        print(f"SAM2 prompts: {len(sam2_prompt_boxes)} box(es)")
    except Exception as e:
        print("SAM2 failed:", repr(e))
        masks_sam2 = None

# 4) Choose which masks to use for the downstream pipeline
if masks_gt_proc is not None:
    masks_proc = masks_gt_proc
    MASK_SOURCE_USED = "gt"
else:
    if MASK_SOURCE_FOR_PIPELINE.lower() == "sam2" and masks_sam2 is not None:
        masks_proc = masks_sam2
        MASK_SOURCE_USED = "sam2"
    elif masks_yolo is not None:
        masks_proc = masks_yolo
        MASK_SOURCE_USED = "yolo"
    else:
        raise RuntimeError("No masks available. Enable RUN_YOLO_MASKS and/or RUN_SAM2_MASKS.")

print("Mask source used for pipeline:", MASK_SOURCE_USED)

# Preview selected masks
mask_rgb = [Image.merge("RGB", (m, m, m)) for m in masks_proc[:12]]
show_grid(mask_rgb, cols=4)



## 11) Compute optical flow for the original video

We compute flows between consecutive **processed** frames.

We compute both directions:
- forward: t -> t+1
- backward: t+1 -> t

Backward flow is used for backward-warping.


In [ ]:

flows_fwd = []  # len = N-1, each (H,W,2)
flows_bwd = []
occlusions = []

for t in tqdm(range(len(frames_proc) - 1), desc="RAFT flows"):
    fwd = raft_flow_between(frames_proc[t], frames_proc[t+1], raft_model, raft_transforms)
    bwd = raft_flow_between(frames_proc[t+1], frames_proc[t], raft_model, raft_transforms)
    flows_fwd.append(fwd)
    flows_bwd.append(bwd)
    occ = compute_occlusion_mask(fwd, bwd, thresh=1.5)
    occlusions.append(occ)

print("Flows computed:", len(flows_fwd))



## 12) Baselines: (A) Trajectory propagation compositor

This baseline:

1. Inpaints the first frame with the replacement object (diffusion)
2. Extracts object patch from the inpainted frame using the mask
3. Applies translation/scale/rotation transforms to the patch across time

It does **not** solve background reveal artifacts (as discussed in the thesis).

We include it primarily as a reference.


In [ ]:

def extract_patch(im_rgb: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray, Tuple[int,int,int,int]]:
    """Return patch RGB, patch mask, and bbox (x0,y0,x1,y1)"""
    ys, xs = np.where(mask > 128)
    if len(xs) == 0 or len(ys) == 0:
        raise ValueError("Empty mask")
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    # pad bbox a bit
    pad = 10
    x0 = max(0, x0 - pad)
    y0 = max(0, y0 - pad)
    x1 = min(im_rgb.shape[1] - 1, x1 + pad)
    y1 = min(im_rgb.shape[0] - 1, y1 + pad)

    patch = im_rgb[y0:y1+1, x0:x1+1].copy()
    pmask = mask[y0:y1+1, x0:x1+1].copy()
    return patch, pmask, (x0, y0, x1, y1)


def apply_affine(patch: np.ndarray, pmask: np.ndarray, scale: float, dtheta: float) -> Tuple[np.ndarray, np.ndarray]:
    h, w = patch.shape[:2]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, math.degrees(dtheta), scale)
    patch_w = cv2.warpAffine(patch, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    mask_w  = cv2.warpAffine(pmask, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return patch_w, mask_w


def composite_patch(bg: np.ndarray, patch: np.ndarray, pmask: np.ndarray, x: int, y: int) -> np.ndarray:
    out = bg.copy()
    h, w = patch.shape[:2]
    x0 = int(round(x))
    y0 = int(round(y))

    # clip
    x1 = min(out.shape[1], x0 + w)
    y1 = min(out.shape[0], y0 + h)
    sx0 = 0
    sy0 = 0
    if x0 < 0:
        sx0 = -x0
        x0 = 0
    if y0 < 0:
        sy0 = -y0
        y0 = 0

    patch_c = patch[sy0:sy0+(y1-y0), sx0:sx0+(x1-x0)]
    mask_c  = pmask[sy0:sy0+(y1-y0), sx0:sx0+(x1-x0)]

    alpha = (mask_c.astype(np.float32) / 255.0)[..., None]
    out[y0:y1, x0:x1] = (alpha * patch_c + (1 - alpha) * out[y0:y1, x0:x1]).astype(np.uint8)
    return out


In [ ]:

# Run baseline A on a short prefix
BASELINE_A_FRAMES = 10

# Inpaint first frame with diffusion
im0 = frames_proc[0]
mask0 = masks_proc[0]

# control image for first frame
control0 = None
if USE_CONTROLNET:
    control0 = make_depth_control(im0) if CONTROLNET_TYPE == "depth" else make_canny_control(im0)

# Safety: make sure control image matches frame size (ControlNet requires this)
if USE_CONTROLNET and control0 is not None and control0.size != im0.size:
    control0 = control0.resize(im0.size, Image.BICUBIC)

print("Sizes -> image:", im0.size, "mask:", mask0.size, "control:", (control0.size if control0 is not None else None))

out0 = pipe_call_inpaint(
    pipe,
    prompt=PROMPT,
    image=im0,
    mask_image=mask0,
    control_image=control0,
    ip_adapter_image=(im0 if USE_IP_ADAPTER else None),
    latents=None,
    generator=gen,
    negative_prompt=NEG_PROMPT,
).images[0]

# extract patch from out0
out0_np = pil_to_np(out0)
mask0_np = np.array(mask0)
patch, pmask, bbox = extract_patch(out0_np, mask0_np)

# trajectory features
feat_prev = trajectory_features(mask0)
if feat_prev is None:
    raise RuntimeError("Failed to extract features from mask0")

baselineA = [out0]
centroid_prev = feat_prev["centroid"]

for t in range(1, min(BASELINE_A_FRAMES, len(frames_proc))):
    feat = trajectory_features(masks_proc[t])
    if feat is None:
        baselineA.append(frames_proc[t])
        continue

    tr = relative_transform(feat_prev, feat)

    # apply affine to patch
    patch_w, pmask_w = apply_affine(patch, pmask, scale=tr["scale"], dtheta=tr["dtheta"])

    # translate using centroid shift (approx)
    dx, dy = tr["dx"], tr["dy"]

    # bbox top-left position in frame 0
    x0, y0, x1, y1 = bbox
    new_x = x0 + dx
    new_y = y0 + dy

    bg = pil_to_np(frames_proc[t])
    comp = composite_patch(bg, patch_w, pmask_w, new_x, new_y)
    baselineA.append(to_pil(comp))

    feat_prev = feat

show_grid(baselineA, cols=4)


## 13) Baseline: (B) Optical flow warping

This reproduces the thesis' idea:

- Compute optical flow between consecutive **original** frames
- Warp the **inpainted** frame forward to approximate the next frame

We implement backward warping using the reverse flow for numerical stability.


In [ ]:

def warp_image_with_bwd_flow(src_rgb: np.ndarray, flow_tgt_to_src: np.ndarray) -> np.ndarray:
    return warp_flow(src_rgb, flow_tgt_to_src)


# Warp out0 to frame1 using backward flow (t1->t0)
if len(frames_proc) >= 2:
    out0_np = pil_to_np(out0)
    flow_1_to_0 = flows_bwd[0]  # computed between frame1->frame0
    warped1 = warp_image_with_bwd_flow(out0_np, flow_1_to_0)
    Image.fromarray(warped1)



## 14) Main method: Controlled diffusion inpainting across frames

We implement:

- **Fixed latent** (base latent)
- **ControlNet** (depth or canny)
- **IP-Adapter** reference image (previous inpainted frame)

This is the strongest *practical* method in the thesis.

### Optional: flow-guided latent search (proposed extension)

If enabled, for each frame we search around the base latent for a candidate latent that makes the inpainted optical flow closer to the original optical flow.

This implements the *minimize flow mismatch* idea without requiring differentiating through the entire diffusion process.


In [ ]:

# Proposed extension toggles
ENABLE_FLOW_MATCH_LATENT_SEARCH = True
FLOW_MATCH_ITERS = 2
FLOW_MATCH_CANDIDATES = 6
FLOW_MATCH_SIGMA = 0.15
FLOW_MATCH_MASK_ONLY = True


In [ ]:

@torch.inference_mode()
def compute_flow_loss(
    prev_inpaint: Image.Image,
    cand_inpaint: Image.Image,
    target_flow: np.ndarray,
    raft_model,
    raft_transforms,
    mask: Optional[np.ndarray] = None,
) -> float:
    """Compute MSE between target_flow and flow(prev_inpaint->cand_inpaint).

    If mask is provided (HxW bool or 0/255), compute loss only in masked region.
    """
    flow_pred = raft_flow_between(prev_inpaint, cand_inpaint, raft_model, raft_transforms)
    diff = (flow_pred - target_flow)

    if mask is not None:
        if mask.dtype != np.bool_:
            mask_bool = mask > 128
        else:
            mask_bool = mask
        # avoid empty mask
        if mask_bool.sum() > 10:
            diff = diff[mask_bool]

    return float(np.mean(diff ** 2))


@torch.inference_mode()
def flow_match_latent_search(
    pipe,
    prompt: str,
    image: Image.Image,
    mask_image: Image.Image,
    control_image: Optional[Image.Image],
    prev_inpaint: Image.Image,
    target_flow: np.ndarray,
    base_latents: torch.Tensor,
    generator: torch.Generator,
    neg_prompt: str,
    iters: int = 2,
    candidates: int = 6,
    sigma: float = 0.15,
    mask_only: bool = True,
) -> Tuple[Image.Image, torch.Tensor, float]:
    """Zero-order search for a latent near base_latents that minimizes optical-flow mismatch.

    Returns: (best_image, best_latents, best_loss)
    """

    best_lat = base_latents.clone()
    # start from base
    out = pipe_call_inpaint(
        pipe,
        prompt=prompt,
        image=image,
        mask_image=mask_image,
        control_image=control_image,
        ip_adapter_image=prev_inpaint,
        latents=best_lat,
        generator=generator,
        negative_prompt=neg_prompt,
    ).images[0]

    mask_np = np.array(mask_image) if mask_only else None
    best_loss = compute_flow_loss(prev_inpaint, out, target_flow, raft_model, raft_transforms, mask=mask_np)

    for _ in range(iters):
        for _k in range(candidates):
            noise = torch.randn_like(best_lat)#, generator=generator)
            cand_lat = best_lat + sigma * noise
            cand = pipe_call_inpaint(
                pipe,
                prompt=prompt,
                image=image,
                mask_image=mask_image,
                control_image=control_image,
                ip_adapter_image=prev_inpaint,
                latents=cand_lat,
                generator=generator,
                negative_prompt=neg_prompt,
            ).images[0]
            loss = compute_flow_loss(prev_inpaint, cand, target_flow, raft_model, raft_transforms, mask=mask_np)
            if loss < best_loss:
                best_loss = loss
                best_lat = cand_lat
                out = cand

    return out, best_lat, best_loss


In [ ]:

# Run the main pipeline
N = len(frames_proc)

results = []
latents_used = []
losses = []

# Precompute model resolution for latents
H, W = frames_proc[0].size[1], frames_proc[0].size[0]  # PIL size is (W,H)
base_latents = make_base_latents(pipe, height=H, width=W, generator=gen)

prev_out = None

for t in tqdm(range(N), desc="Inpainting"):
    frame = frames_proc[t]
    mask = masks_proc[t]

    if USE_CONTROLNET:
        control = make_depth_control(frame) if CONTROLNET_TYPE == "depth" else make_canny_control(frame)
    else:
        control = None

    if t == 0:
        out = pipe_call_inpaint(
            pipe,
            prompt=PROMPT,
            image=frame,
            mask_image=mask,
            control_image=control,
            ip_adapter_image=(frame if USE_IP_ADAPTER else None),
            latents=base_latents,
            #generator=gen,
            negative_prompt=NEG_PROMPT,
        ).images[0]
        results.append(out)
        latents_used.append(base_latents.detach().cpu())
        losses.append(float("nan"))
        prev_out = out
        continue

    # target optical flow between original frames t-1 -> t
    target_flow = flows_fwd[t-1]

    if ENABLE_FLOW_MATCH_LATENT_SEARCH:
        out, lat_t, loss = flow_match_latent_search(
            pipe,
            prompt=PROMPT,
            image=frame,
            mask_image=mask,
            control_image=control,
            prev_inpaint=prev_out,
            target_flow=target_flow,
            base_latents=base_latents,
            generator=gen,
            neg_prompt=NEG_PROMPT,
            iters=FLOW_MATCH_ITERS,
            candidates=FLOW_MATCH_CANDIDATES,
            sigma=FLOW_MATCH_SIGMA,
            mask_only=FLOW_MATCH_MASK_ONLY,
        )
        losses.append(loss)
        latents_used.append(lat_t.detach().cpu())

    else:
        out = pipe_call_inpaint(
            pipe,
            prompt=PROMPT,
            image=frame,
            mask_image=mask,
            control_image=control,
            ip_adapter_image=prev_out,
            latents=base_latents,
            generator=gen,
            negative_prompt=NEG_PROMPT,
        ).images[0]
        losses.append(float("nan"))
        latents_used.append(base_latents.detach().cpu())

    results.append(out)
    prev_out = out

print("Done. Frames:", len(results))
show_grid(results[:12], cols=4)



## 15) Export video


In [ ]:

def save_video(frames_pil: List[Image.Image], out_path: str, fps: int = 10):
    ensure_dir(os.path.dirname(out_path))
    writer = imageio.get_writer(out_path, fps=fps, codec="libx264", quality=8)
    try:
        for im in frames_pil:
            writer.append_data(np.array(im.convert("RGB")))
    finally:
        writer.close()


out_video = os.path.join(OUT_DIR, "result.mp4")
save_video(results, out_video, fps=10)
print("Saved:", out_video)



## 16) Quantitative checks

We compute:

- Flow-matching loss (already collected if enabled)
- Simple temporal L1 difference between consecutive inpainted frames

These are not perfect metrics, but they are useful for debugging.


In [ ]:

# Temporal L1 between consecutive generated frames
l1s = []
for t in range(1, len(results)):
    a = np.array(results[t-1].convert("RGB")).astype(np.float32) / 255.0
    b = np.array(results[t].convert("RGB")).astype(np.float32) / 255.0
    l1s.append(float(np.mean(np.abs(a - b))))

print("Temporal L1 mean:", float(np.mean(l1s)))

if ENABLE_FLOW_MATCH_LATENT_SEARCH:
    # ignore nan for first
    vals = [x for x in losses[1:] if np.isfinite(x)]
    print("Flow-match MSE mean:", float(np.mean(vals)))


## 16b) SAM2 vs YOLOv8 mask comparison + SSIM metrics

Since our dataset has **no ground-truth masks**, we compare the two mask sources **against each other**:

- **Mask IoU / Dice** (agreement between binary masks)
- **Mask SSIM** (structural similarity between the two masks)
- (Optional) **Temporal SSIM** on the *input* frames and on the *generated* frames (if you ran the inpainting step)

We also export a **side-by-side mask overlay video** to visually compare YOLO vs SAM2.


In [ ]:
from skimage.metrics import structural_similarity as ssim
import pandas as pd

def binarize_mask_pil(m: Image.Image, thr: int = 128) -> np.ndarray:
    arr = np.array(m.convert("L"))
    return (arr > thr).astype(np.uint8)

def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = float(np.logical_and(a, b).sum())
    union = float(np.logical_or(a, b).sum())
    return inter / union if union > 0 else 1.0

def mask_dice(a: np.ndarray, b: np.ndarray) -> float:
    inter = float(np.logical_and(a, b).sum())
    denom = float(a.sum() + b.sum())
    return (2.0 * inter / denom) if denom > 0 else 1.0

def mask_ssim(a: np.ndarray, b: np.ndarray) -> float:
    # a,b are binary {0,1}
    return float(ssim(a.astype(np.float32), b.astype(np.float32), data_range=1.0))

def temporal_ssim(frames: List[Image.Image]) -> float:
    vals = []
    for t in range(1, len(frames)):
        a = np.array(frames[t-1].convert("L"))
        b = np.array(frames[t].convert("L"))
        vals.append(float(ssim(a, b, data_range=255)))
    return float(np.mean(vals)) if vals else float("nan")

# --- 1) Mask agreement metrics (YOLO vs SAM2) ---
if masks_yolo is None or masks_sam2 is None:
    print("Need both masks_yolo and masks_sam2 to compute agreement metrics.")
else:
    rows = []
    for t, (my, ms) in enumerate(zip(masks_yolo, masks_sam2)):
        a = binarize_mask_pil(my)  # {0,1}
        b = binarize_mask_pil(ms)
        rows.append({
            "frame_idx": t,
            "iou": mask_iou(a, b),
            "dice": mask_dice(a, b),
            "ssim": mask_ssim(a, b),
            "area_yolo": int(a.sum()),
            "area_sam2": int(b.sum()),
        })

    df = pd.DataFrame(rows)
    print("Mask agreement summary:")
    display(df[["iou","dice","ssim"]].describe())

    out_csv = os.path.join(OUT_DIR, "mask_metrics_sam2_vs_yolo.csv")
    df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

# --- 2) Temporal SSIM on the input frames ---
print("Temporal SSIM (input frames_proc):", temporal_ssim(frames_proc))

# --- 3) Temporal SSIM on generated frames (if available) ---
if "results" in globals() and isinstance(results, list) and len(results) > 1:
    print("Temporal SSIM (generated results):", temporal_ssim(results))
else:
    print("No `results` frames found (skip generated Temporal SSIM).")

# --- 4) Export a side-by-side mask overlay video (YOLO vs SAM2) ---
def overlay_mask(frame_rgb: np.ndarray, mask01: np.ndarray, color_rgb=(0,255,0), alpha: float = 0.5) -> np.ndarray:
    out = frame_rgb.copy()
    color = np.array(color_rgb, dtype=np.float32)
    idx = mask01.astype(bool)
    out[idx] = (1 - alpha) * out[idx].astype(np.float32) + alpha * color
    return out.astype(np.uint8)

if masks_yolo is not None and masks_sam2 is not None:
    compare_frames = []
    for t in range(len(frames_proc)):
        fr = np.array(frames_proc[t].convert("RGB"))
        a = binarize_mask_pil(masks_yolo[t])
        b = binarize_mask_pil(masks_sam2[t])

        yolo_ov = overlay_mask(fr, a, color_rgb=(0,255,0), alpha=0.45)
        sam2_ov = overlay_mask(fr, b, color_rgb=(255,0,0), alpha=0.45)

        side = np.concatenate([yolo_ov, sam2_ov], axis=1)
        compare_frames.append(Image.fromarray(side))

    out_cmp = os.path.join(OUT_DIR, "mask_compare_yolo_vs_sam2.mp4")
    save_video(compare_frames, out_cmp, fps=10)
    print("Saved:", out_cmp)
else:
    print("Skip mask overlay video (need both masks_yolo and masks_sam2).")


## 17) Notes / next research steps

If you want to go beyond this notebook:

1. Improve mask quality:
   - Tune YOLO class/conf thresholds (and track multiple objects).
   - Use SAM2 with manual prompts (points/boxes) for harder cases, or track more objects from YOLO prompts.
   - If you can label a small subset, evaluate against GT masks (IoU/F-score).
2. Use a stronger motion model for non-rigid motion than centroid/diameter geometry.
3. Replace zero-order latent search with:
   - learned latent predictor (small MLP)
   - gradient-based latent optimization (requires custom differentiable pipeline)
4. Move from SD1.5 to SDXL inpainting for higher resolution.
